# <span style="color:#ff8c00">04 · Modern encoder benchmark</span>

Modern frozen image encoders (SigLIP 2, Perception Encoder, C-RADIOv4, CLIP ViT-L/14, Qwen3-VL-Embedding) crossed with modern question encoders (Qwen3-Embedding 0.6B and 4B, Qwen3-VL-Embedding text side, and BERT-base as the control) and both fusion heads (`attention`, `product`) on the 3,129-answer list, plus the classical winner re-run as a bridge and the Qwen token-state study.

Modes (`MODE` below, in run order): `run` (evaluate every cell of one image encoder at a time, then delete its cached features) · `report` (val pivot tables, bridge comparison, bootstrap intervals and the token-state comparison).

In [ ]:
MODE = "run"          # run | report
IMAGES = []           # image encoder keys; empty = every tier-1 entry of configs/vqa_lab.yaml (modern.image)
TEXTS = ["qwen3_emb_0.6b", "qwen3_emb_4b", "qwen3vl_emb_2b_text", "bert"]   # question encoders (bert = the classical bridge arm)
HEADS = ["attention", "product"]   # attention first: its grid extraction also writes the pooled vectors, so each encoder is extracted once
ANSWERS = 3129        # the 3,129 most frequent majority answers of the training rows
BRIDGE = [("convnext_base.clip_laion2b", "bert", "attention")]   # the classical winner, re-run with this answer list
BRIDGE_CLASSICAL = 60.27   # val % of the bridge model with the classical 1,000-answer list
TOKEN_IMAGES = ["qwen3vl_emb_2b", "pe_lang_l"]   # token-state study (2026-10-21): two image encoders
TOKEN_TEXTS = ["qwen3_emb_0.6b_tokens", "qwen3_emb_4b_tokens"]   # Qwen last-layer token states through the BiLSTM
LIMIT = None          # smoke tests (question subsample)
EPOCHS = 50
PATIENCE = 4
GPU_STATES = True     # BERT token states on the GPU (the sentence vectors always are)

# <span style="color:#ff8c00">Setup</span>

Paths, the resumable run table of this benchmark, the encoder list and the data context (3,129-answer list).

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # before CUDA starts: less fragmentation, same numbers

import vqa_lab
from vqa_lab import benchmark as B
from vqa_lab.config import load_encoders, load_paths
from vqa_lab.data import fetch_hub, read_table
from vqa_lab.report import write_report
from vqa_lab.text import hub_repo

paths = load_paths()
table = B.RunTable(paths, paths.runs_dir / ("modern" + ("_smoke" if LIMIT else "")))   # resumes from its parquet table
images = IMAGES or [e["key"] for e in load_encoders(paths, "modern_image") if e.get("tier") == 1]
ctx = vqa_lab.context(limit=LIMIT, answers=ANSWERS) if MODE != "report" else None
RUN = dict(epochs=EPOCHS, patience=PATIENCE, gpu_states=GPU_STATES)
print(len(images), "image encoders x", len(TEXTS), "question encoders x", len(HEADS), "heads")

# <span style="color:#ff8c00">Question encoders on disk</span>

Downloaded once into `data/raw_files/models/<org>/<name>` (the image encoders download themselves on first use).

In [ ]:
for arm in TEXTS + TOKEN_TEXTS:
    if repo := hub_repo(arm):
        fetch_hub(repo, paths.raw_dir / "models" / repo, patterns=["*.json", "*.safetensors", "*.txt", "*.jinja"])

# <span style="color:#ff8c00">1 · One image encoder at a time</span>

Finished cells are reused; a failing encoder is logged and skipped; the features are deleted after each encoder.

In [ ]:
if MODE == "run":   # every cell of an image encoder in one visit, so its features are extracted once
    plan = {key: [(t, h) for t in TEXTS for h in HEADS] for key in images}   # one question arm on the GPU at a time: both heads, then the next arm
    for key in TOKEN_IMAGES:   # token-state study
        plan.setdefault(key, []).extend((t, h) for t in TOKEN_TEXTS for h in HEADS)
    for key, text, head in BRIDGE:
        plan.setdefault(key, []).append((text, head))
    for key, cells in plan.items():
        table.visit(ctx, key, cells, MODE, **RUN)

# <span style="color:#ff8c00">2 · Report</span>

Every cell, the bridge to the classical table, 95 % image-bootstrap intervals of the best runs, and the token-state comparison.

The record of this benchmark is `artifacts/runs/modern/results.parquet` with `failed.csv`. The intervals read the per-question files in its `scores/` folder, which is only a cache: delete it once the report is done.

The report also merges every finished benchmark into one file, `benchmark_results.md`, next to the notebooks.

In [ ]:
if MODE == "report":
    res = table.results
    if res.empty:
        print("No runs yet: run the benchmark first.")

if MODE == "report" and len(res):
    res = B.trained_runs(res)
    print("val % (rows: image encoder; columns: head, question encoder):")
    display(res.pivot_table(index="backbone", columns=["fusion", "text"], values="val").round(2))
    print(f"\nBridge: the classical winner with the 3,129-answer list (classical 1,000-answer result: {BRIDGE_CLASSICAL}):")
    display(res[res["backbone"].isin([k for k, _, _ in BRIDGE])][["backbone", "text", "fusion", "val"]].round(2))
    if not LIMIT:
        print("\nTop runs with 95 % image-bootstrap intervals and the paired gap to the best run:")
        display(B.confidence_table(table.results, table.run_dir / "scores", read_table(paths, "val")["image_id"].to_numpy(), top=10))
        print("\nAll benchmarks merged into:", write_report(paths))
    print("\nToken-state study (like-for-like):")
    display(B.token_table(table.results))